In [ ]:
############################################################################
############################ POLYMODELS DATA ###############################
############################################################################


# PURPOSE
############################################################################

# A notebook to prepare the factor set for polymodels



# HISTORY
############################################################################

# V1 is the initial version 



# IMPORTS 
############################################################################

# System
import os
import warnings
import time
from datetime import timedelta
import logging
logging.basicConfig()
logging.getLogger().setLevel(logging.INFO)
LOGGER = logging.getLogger(__name__)
import copy
from tqdm import tqdm

# Pandas and Python
import pandas as pd
idx = pd.IndexSlice
pd.options.display.float_format = '{:.4f}'.format
from collections import OrderedDict
from multiprocessing import Pool
from functools import partial
import numpy as np
import statsmodels.api as sm
from itertools import combinations
from dateutil.relativedelta import relativedelta

# Graphic Libraries
import cufflinks as cf
cf.go_offline()
from IPython.core.display import display, HTML
from IPython.display import clear_output
from plotly import tools
import plotly.offline as py
import plotly.figure_factory as ff
from plotly import graph_objs as go




In [ ]:
# NOTEBOOK PARAMETERS
############################################################################

# Define cache paths
data_path = "/home/tbarrau/SPEC/"



In [ ]:
# LOAD DATA 
############################################################################

# Load data
data = pd.read_excel(
    f"{data_path}230104 Factor Set.xlsx", 
    sheet_name=1,
    header=[0,1])


In [ ]:
# Load data dictionnary
data_dictionnary = pd.read_excel(
    f"{data_path}230210 Factor Set - Dictionnary (bbg only).xlsx", 
)
data_dictionnary.index = data_dictionnary["SYMBOL"]
data_dictionnary = data_dictionnary.drop("SYMBOL", axis = 1)



In [ ]:
# REFORMAT DATA 
############################################################################

# Reformat index
data.index = data.iloc[:,0]
data = data.iloc[:,1:]
data.index.name = 'Date'


In [ ]:
# Extract px_last and total returns columns
px_last_columns = []
total_returns_columns = []

for i in data.columns:
    if i[1] == "PX_LAST":
        px_last_columns.append(i)
    elif i[1] == "PX_OPEN":
        pass
    else:
        total_returns_columns.append(i)
        
# Separate DataFrames
px_last = data[px_last_columns]
px_last.columns = [i[0] for i in px_last.columns]

total_returns_index = data[total_returns_columns]
total_returns_index.columns = [i[0] for i in total_returns_index.columns]


In [ ]:
# Convert string numbers to float format
print("Processing total returns index")
total_returns_index = total_returns_index.replace('#N/A Invalid Security', np.nan)
for i in tqdm(total_returns_index.columns):
    total_returns_index[i] = total_returns_index[i].astype(float)
    
print("Processing px last")
px_last = px_last.replace('#N/A Invalid Security', np.nan)
for i in tqdm(px_last.columns):
    px_last[i] = px_last[i].astype(float)
    

In [ ]:
# Forward fill reasonnably
total_returns_index = total_returns_index.ffill(limit=21*3)
px_last = px_last.ffill(limit=21*3)


In [ ]:
# Create instruments returns, total returns index based
returns = total_returns_index.replace(0, np.nan).sub(
    total_returns_index.shift(1)).div(
    total_returns_index.shift(1).abs())
# /!\ prices can be negative (e.g. oil)

# Fill blanks with px_last based returns
px_last_returns = px_last.replace(0, np.nan).sub(
    px_last.shift(1)).div(
    px_last.shift(1).abs())
returns = returns.replace(0, np.nan)
returns = returns.fillna(px_last_returns)

# Deal with the special case of rates:
# For rates, we do consider the difference in rate,
# we don't want to divide by anything, as a rate can be very close to 0
rates_columns = data_dictionnary['Nature'].where(
    data_dictionnary['Nature'] == 'Rate').dropna().index
returns[rates_columns] = (px_last/100).diff()[rates_columns]

# Deal with the special case of volatility (VIX)
# related indicators: we don't really want to use the volatility change,
# rather the level
volatility_columns = ["UX1 Index", "VXEEM Index", "VXEFA Index", "VXEWZ Index",
                      "VXFXI Index", "VXGDX Index", "VXSLV Index"]
returns[volatility_columns] = px_last[volatility_columns]



In [ ]:
# Remove empty lines
returns = returns.replace(0, np.nan).dropna(how='all')

# Remove empty columns
returns = returns.replace(0, np.nan).dropna(how='all', axis=1)


In [ ]:
# Remove infinite returns
returns = returns.replace(np.inf, np.nan)
returns = returns.replace(-np.inf, np.nan)


In [ ]:
# Remove aberrant returns (handmade selection)
returns.loc["2004-03-23", "F3DIND Index"] = np.nan
returns.loc["2015-04-14":, "BSET Index"] = np.nan
returns.loc["2008-09-01":, "MXRU0UT Index"] = np.nan
returns.loc["2010-12-02":, "MXID0HC Index"] = np.nan
returns.loc['2018-06-04', 'MXEG0IN Index'] = np.nan
returns.loc['2001-06-20', 'SASEELE Index'] = np.nan
returns.loc['2000-01-12', 'SASETISI Index'] = np.nan
returns.loc['2010-08-16', 'JMDEUR Curncy'] = np.nan
returns.loc['2001-12-20', 'JFINX Index'] = np.nan
returns.loc['2009-12-02', 'MXCO0CS Index'] = np.nan
returns.loc['2011-11-17', 'JMDEUR Curncy'] = np.nan
returns.loc['2018-03-01', 'MXEG0CS Index'] = np.nan
returns.loc['2014-03-03', 'MXDK0MT Index'] = np.nan
returns.loc['2009-11-30', 'F3REAL Index'] = np.nan
returns.loc['2016-01-04', 'SASETISI Index'] = np.nan
returns.loc['2005-05-02', 'MXID0EN Index'] = np.nan
returns.loc['2003-04-01', 'MXZA0EN Index'] = np.nan
returns.loc['2007-02-27', 'DWGFN Index'] = np.nan
returns.loc['2010-12-02', 'MXBE0IN Index'] = np.nan
returns.loc['2002-07-25', 'SASEELE Index'] = np.nan
returns.loc['2002-07-24', 'SASETCSI Index'] = np.nan
returns.loc['2012-11-23', 'NGSEFB10 Index'] = np.nan
returns.loc['2011-08-02', 'MXDK0CD Index'] = np.nan
returns.loc['2022-03-10', 'JMSMX Index'] = np.nan
returns.loc['2002-05-28', 'RGUSMS Index'] = np.nan
returns.loc['2022-02-28', 'MXRU0CS Index'] = np.nan
returns.loc['2019-08-12', 'MXAR0FN Index'] = np.nan
returns.loc['2019-08-12', 'MXAR0UT Index'] = np.nan
returns.loc['2004-09-27', 'XUMAL Index'] = np.nan
returns.loc['2010-11-30', 'MXBR0HC Index'] = np.nan
returns.loc['2015-12-01', 'MXAR0CS Index'] = np.nan
returns.loc['2002-10-02', 'F3INFT Index'] = np.nan
returns.loc['2004-08-05', 'BOXBBC9V Index'] = np.nan
returns.loc['2004-09-07', 'BOXBBC9V Index'] = np.nan
returns.loc['2004-12-20', 'DWMFOG Index'] = np.nan
returns.loc['2022-02-24', 'MXRU0CD Index'] = np.nan
returns.loc['2016-10-19', 'F3INFT Index'] = np.nan
returns.loc['2005-06-03', 'BOXBBC9V Index'] = np.nan
returns.loc['2001-11-15', 'MXNZ0HC Index'] = np.nan
returns.loc['2008-04-24', 'JMDEUR Curncy'] = np.nan
returns.loc['2019-07-24', 'EGXREAL Index'] = np.nan
returns.loc['2019-04-30', 'EGXREAL Index'] = np.nan
returns.loc['2019-05-02', 'EGXREAL Index'] = np.nan
returns.loc['2019-07-02', 'EGXREAL Index'] = np.nan
returns.loc['2005-06-01', 'MXEG0MT Index'] = np.nan
returns.loc['2008-11-17', 'MXRU0CD Index'] = np.nan
returns.loc['2008-03-25', 'MXEG0IN Index'] = np.nan
returns.loc['2019-02-04', 'EGXREAL Index'] = np.nan
returns.loc['2014-02-17', 'NGSEOG5 Index'] = np.nan
returns.loc['2005-05-27', 'BOXBBC9V Index'] = np.nan
returns.loc['2011-03-01', 'MXDK0CD Index'] = np.nan
returns.loc['2019-08-14', 'EGXREAL Index'] = np.nan
returns.loc['2019-06-10', 'EGXREAL Index'] = np.nan
returns.loc['2011-10-11', 'NGSEOG5 Index'] = np.nan
returns.loc['2011-10-12', 'NGSEIN10 Index'] = np.nan
returns.loc['2008-09-02', 'BOXBBC9V Index'] = np.nan
returns.loc['2005-05-03', 'BOXBBC9V Index'] = np.nan
returns.loc['2008-09-09', 'BOXBBC9V Index'] = np.nan
returns.loc['2002-01-17', 'MXAR0FN Index'] = np.nan
returns.loc['2003-09-18', 'BOXBBC9V Index'] = np.nan
returns.loc['2004-01-05', 'BOXBBC9V Index'] = np.nan
returns.loc['2002-05-28', 'RGUSSS Index'] = np.nan
returns.loc['2002-05-27', 'RGUSSS Index'] = np.nan
returns.loc['2008-12-12', 'NGSEFB10 Index'] = np.nan
returns.loc['2011-01-31', 'BOXBBC9V Index'] = np.nan
returns.loc['2011-08-05', 'BOXBBC9V Index'] = np.nan
returns.loc['2002-07-02', 'SASETISI Index'] = np.nan
returns.loc['2016-11-15', 'SENCYX Index'] = np.nan
returns.loc['2018-06-01', 'MXAR0CD Index'] = np.nan
returns.loc['2000-08-03', 'FNMR Index'] = np.nan
returns.loc['2000-12-25', 'JN1 Comdty'] = np.nan
returns.loc['2016-09-19', 'BOXBBC9V Index'] = np.nan
returns.loc['2000-05-23', 'BSETCD Index'] = np.nan
returns.loc['2016-06-21', 'NGNJPY Curncy'] = np.nan
returns.loc['2017-08-11', 'DWMFHC Index'] = np.nan
returns.loc['2022-02-28', 'MXRU0EN Index'] = np.nan
returns.loc['2000-06-12', 'BSETCD Index'] = np.nan
returns.loc['2000-09-27', 'DWMFCG Index'] = np.nan
returns.loc['2012-07-03', 'BOXBBC9V Index'] = np.nan
returns.loc['2021-05-28', 'MXGR0FN Index'] = np.nan
returns.loc['2013-11-27', 'MXGR0FN Index'] = np.nan
returns.loc['2020-05-11', 'DWMFTC Index'] = np.nan
returns.loc['2002-07-24', 'SASETCSI Index'] = np.nan
returns.loc['2002-07-25', 'SASETCSI Index'] = np.nan
returns.loc['2002-07-24', 'SASEELE Index'] = np.nan
returns.loc['2002-07-25', 'SASEELE Index'] = np.nan
returns.loc['2006-08-23', 'MXAR0IN Index'] = np.nan
returns.loc['2021-12-01', 'MXAR0TC Index'] = np.nan
returns.loc['2010-12-02', 'MXTH0IN Index'] = np.nan
returns.loc['2007-02-27', 'DWGFN Index'] = np.nan
returns.loc['2007-02-28', 'DWGFN Index'] = np.nan
returns.loc['2021-12-01', 'MXAR0FN Index'] = np.nan
returns.loc['2013-06-03', 'MXMX0HC Index'] = np.nan
returns.loc['2014-06-02', 'MXDK0CD Index'] = np.nan
returns.loc['2013-11-27', 'MXGR0UT Index'] = np.nan
returns.loc['2011-11-16', 'JMDEUR Curncy'] = np.nan
returns.loc['2011-11-17', 'JMDEUR Curncy'] = np.nan
returns.loc['2010-08-13', 'JMDEUR Curncy'] = np.nan
returns.loc['2010-08-16', 'JMDEUR Curncy'] = np.nan
returns.loc['2016-03-01', 'MXAR0CD Index'] = np.nan
returns.loc['2020-06-01', 'MXIT0IT Index'] = np.nan
returns.loc['2001-12-31', 'JFINX Index'] = np.nan
returns.loc['2000-01-12', 'SASETISI Index'] = np.nan
returns.loc['2000-01-13', 'SASETISI Index'] = np.nan
returns.loc['2001-06-20', 'SASEELE Index'] = np.nan
returns.loc['2001-06-21', 'SASEELE Index'] = np.nan
returns.loc['2013-05-02', 'MXGR0TC Index'] = np.nan
returns.loc['2021-05-28', 'MXRU0CD Index'] = np.nan
returns.loc['2006-08-23', 'MXIT0HC Index'] = np.nan
returns.loc['2003-12-11', 'FSTUT Index'] = np.nan
returns.loc['2022-06-01', 'MXAR0CD Index'] = np.nan
returns.loc['2012-06-01', 'MXAT0IN Index'] = np.nan
returns.loc['2013-11-27', 'MXGR0MT Index'] = np.nan
returns.loc['2013-11-27', 'MXGR0EN Index'] = np.nan
returns.loc['2004-09-28', 'XUMAL Index'] = np.nan
returns.loc['2003-05-29', 'F3CONGI Index'] = np.nan
returns.loc['2015-12-01', 'MXGR0UT Index'] = np.nan
returns.loc['2016-12-01', 'MXAR0UT Index'] = np.nan
returns.loc['2002-05-27', 'RGUSMS Index'] = np.nan
returns.loc['2022-06-01', 'MXAR0UT Index'] = np.nan
returns.loc['2012-12-03', 'MXTH0IN Index'] = np.nan
returns.loc['2012-01-03', 'NGSEFB10 Index'] = np.nan
returns.loc['2022-03-11', 'JMSMX Index'] = np.nan
returns.loc['2008-03-26','MXEG0IN Index'] = np.nan
returns.loc['2014-02-18','NGSEOG5 Index'] = np.nan
returns.loc['2017-12-01','MXPL0IT Index'] = np.nan
returns.loc['2001-03-19','DJUSCL Index'] = np.nan
returns.loc['2008-04-25','JMDEUR Curncy'] = np.nan
returns.loc['2006-08-23','MXMX0EN Index'] = np.nan
returns.loc['2018-09-03','MXGR0EN Index'] = np.nan
returns.loc['2021-11-17','MXGR0UT Index'] = np.nan
returns.loc['2016-10-31':'2016-12-21','EB03/3M Index'] = np.nan
returns.loc['2018-03-13':'2018-04-12','EB03/3M Index'] = np.nan
returns.loc['2001-01-04','AEDUSD Curncy'] = np.nan
returns.loc['2001-01-05','AEDUSD Curncy'] = np.nan
returns.loc['2002-12-06','AEDUSD Curncy'] = np.nan
returns.loc['2002-12-09','AEDUSD Curncy'] = np.nan
returns.loc['2005-05-23','AEDUSD Curncy'] = np.nan
returns.loc['2005-05-24','AEDUSD Curncy'] = np.nan
returns.loc['2001-01-01','DKKEUR Curncy'] = np.nan
returns.loc['2001-01-02','DKKEUR Curncy'] = np.nan
returns.loc['2000-11-20','MAGY3YR Index'] = np.nan
returns.loc['2000-11-21','MAGY3YR Index'] = np.nan
returns.loc['2005-07-21','CNYUSD Curncy'] = np.nan
returns.loc['2022-11-22','MICXRU20 Index'] = np.nan
returns.loc['2022-11-22','MICXRU10 Index'] = np.nan
returns.loc['2022-11-22','MICXRU5Y Index'] = np.nan
returns.loc['2022-11-22','MICXRU3Y Index'] = np.nan
returns.loc['2009-04-07':'2009-07-07':,'NDDR1T Curncy'] = np.nan
returns.loc['2019-06-06':'2019-06-10':,'BDTEUR Curncy'] = np.nan
returns.loc['2019-06-06':'2019-06-07':,'BDTUSD Curncy'] = np.nan


In [ ]:
# SAVE DATA 
############################################################################

returns.to_csv(f"{data_path}230216_returns.csv")

In [ ]:
pd.read_csv(f"{data_path}230216_returns.csv")